# End-to-end results

Builds the paper's main results table from the `ramp` pipeline of every device: Top-1
accuracy and median latency of FP32, uniform INT8 and the candidate each policy selects.

In [ ]:
# Parameters
ENTITY = None            # None: the default entity of the logged-in account
PROJECT = "ramp"
# Runs are picked by tag: the latest one per (device, model). Tag the runs to report.
TAG = "current"
# Or pin them explicitly: {(device host, model name): "<run id>"}. Overrides the tag.
RUNS = {}

# Run names are "<model name>-<host>". Order here is the column order of the table.
DEVICES = {
    "MacBook-Air-de-David.local": "Apple M1",
    "tt-rpi51": "Raspberry Pi 5 (Cortex-A76)",
}
MODELS = {
    "gluon_resnet18_v1b": "ResNet-18",
    "efficientnet_b0": "EffNet-B0",
    "convnext_tiny": "ConvNeXt-T",
    "tiny_vit_21m_224.in1k": "TinyViT",
}
# Row order. A policy no run evaluated is left blank.
POLICIES = ["FP32", "INT8 unif.", "STD", "HAWQ-V2", "RAMP (JSD)", "JSD + Latency"]
# A policy collapses when it falls more than this many points below FP32.
COLLAPSE_POINTS = 15.0

In [ ]:
import logging
import os
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import rootutils

root = rootutils.setup_root(".", indicator=".project-root", pythonpath=True, cwd=True)

os.environ.setdefault("WANDB_SILENT", "true")  # no download progress in the outputs
import wandb

from src.flows.ramp.final_results import select_candidates

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s", force=True)
api = wandb.Api()
entity = ENTITY or api.default_entity

## Runs

One run per device and model.

In [ ]:
def _host(run):
    model = run.config["model"]["model_name"]
    return run.name[len(model) + 1:] if run.name.startswith(model + "-") else None


chosen = {}
if RUNS:
    for key, run_id in RUNS.items():
        chosen[key] = api.run(f"{entity}/{PROJECT}/{run_id}")
else:
    # Newest first, so the first run seen for a (device, model) is the latest.
    for run in api.runs(f"{entity}/{PROJECT}", filters={"tags": TAG}, order="-created_at"):
        key = (_host(run), run.config["model"]["model_name"])
        if key[0] in DEVICES and key[1] in MODELS:
            chosen.setdefault(key, run)

used = pd.DataFrame(
    [
        {
            "device": DEVICES[host],
            "model": MODELS[model],
            "run": run.id,
            "created": run.created_at[:16],
            "intra-op threads": run.config.get("session_options", {}).get("intra_op_num_threads"),
            "metrics": ", ".join(run.config.get("final_results", {}).get("metrics", {})),
        }
        for (host, model), run in chosen.items()
    ]
)
missing = [(DEVICES[h], MODELS[m]) for h in DEVICES for m in MODELS if (h, m) not in chosen]
if missing:
    print("No run for:", missing)
used.sort_values(["device", "model"]).reset_index(drop=True)

## Selected candidate per policy

`select_candidates` keeps FP32 and uniform INT8 as they are and, for every other policy,
the elbow of its accuracy-latency sweep anchored on FP32.

In [ ]:
cache = Path(tempfile.mkdtemp(prefix="ramp_table_"))


def _sweep(run):
    """The run's full sweep: final_results.csv of its final-results artifact."""
    for artifact in run.logged_artifacts():
        if artifact.name.startswith("final-results"):
            folder = artifact.download(root=str(cache / run.id))
            return pd.read_csv(Path(folder) / "final_results.csv")
    raise LookupError(f"Run {run.id} has no final-results artifact")


rows = []
for (host, model), run in chosen.items():
    selected = select_candidates(_sweep(run))
    fp32 = selected.loc[selected["policy"] == "FP32", "acc_pct"].iloc[0]
    for _, r in selected.iterrows():
        rows.append(
            {
                "device": DEVICES[host],
                "model": MODELS[model],
                "policy": r["policy"],
                "config": r["name"],
                "acc": r["acc_pct"],
                "lat": r["latency_ms"],
                "speedup": r["speedup"],
                "int8_nodes": r["n_quantized_layers"],
                "collapse": r["acc_pct"] < fp32 - COLLAPSE_POINTS,
            }
        )
results = pd.DataFrame(rows)
results["device"] = pd.Categorical(results["device"], list(DEVICES.values()), ordered=True)
results["model"] = pd.Categorical(results["model"], list(MODELS.values()), ordered=True)
results["policy"] = pd.Categorical(results["policy"], POLICIES, ordered=True)
results = results.sort_values(["device", "model", "policy"]).reset_index(drop=True)
results.round(2)

## Final table

Accuracy in %, latency in ms. Collapsed cells are shaded.

In [ ]:
wide = results.pivot_table(
    index="policy", columns=["device", "model"], values=["acc", "lat"], observed=True
)
wide = wide.reorder_levels([1, 2, 0], axis=1)
columns = [
    (d, m, v) for d in DEVICES.values() for m in MODELS.values() for v in ("acc", "lat")
    if (d, m, v) in wide.columns
]
wide = wide.reindex(index=[p for p in POLICIES if p in wide.index], columns=pd.MultiIndex.from_tuples(columns))
collapsed = results.pivot_table(
    index="policy", columns=["device", "model"], values="collapse", aggfunc="first", observed=True
)


def _shade(frame):
    styles = pd.DataFrame("", index=frame.index, columns=frame.columns)
    for device, model, value in frame.columns:
        if value != "Acc" or (device, model) not in collapsed.columns:
            continue
        for policy in frame.index:
            if bool(collapsed.loc[policy, (device, model)]):
                styles.loc[policy, (device, model, value)] = "background-color: #d1d1d1"
    return styles


wide.rename(columns={"acc": "Acc", "lat": "Lat"}, level=2).style.format("{:.1f}", na_rep="--").apply(
    _shade, axis=None
)